# quickstart

Fetch HB1603 raw files from NCEI, compute Sv and plot an echogram.

**Author:** <Your Name>
**Version:** 1.0

In [ ]:
# Uncomment the lines below to install required packages.
# %pip install git+https://github.com/BLayman-NOAA/AA-SI_Utils.git
# %pip install echopype>=0.6.0
# %pip install git+https://github.com/BLayman-NOAA/AA-SI_Visualization.git

In [ ]:
from aa_si_utils.data_retrieval import query_ncei_data
from aa_si_utils.utils import read_raw_files_to_stores
from aa_si_utils.utils import combine_raw_stores
from echopype.calibrate import compute_Sv
from aa_si_visualization.echogram import plot_sv_echogram


In [ ]:
cruise_name = 'HB1603'  # str
file_time_start = '2016-07-25T20:58:00'  # str
file_time_end = '2016-07-25T21:45:00'  # str
sonar_model = 'EK60'  # str
max_depth = 1000.0  # float

## Step: `query_ncei`
**Op:** `query_ncei_data`

Query an NCEI acoustic data catalog for file listings within a time window. Returns a list of result records (file metadata), not the files themselves.

**Parameters:**
- `file_time_start` str - ISO 8601 start of window, e.g. '2016-07-25T20:58'
- `file_time_end` str - ISO 8601 end of window
- `filters` dict - Key-value filters applied to NCEI metadata (e.g. CRUISE_NAME)
- `file_names` list - Keep only these files, matched on the D{date}-T{time} stamp in the name. A name the query did not return is an error. Empty keeps everything.


In [ ]:
# --- Parameters ---
_query_ncei__file_time_start = '2016-07-25T20:58:00'
_query_ncei__file_time_end = '2016-07-25T21:45:00'

_result = query_ncei_data(
    file_time_start=_query_ncei__file_time_start,
    file_time_end=_query_ncei__file_time_end,
    filters={'CRUISE_NAME': 'HB1603'},
)
ncei_results = _result['records']
query_label = _result['query_label']
raw_urls = _result['raw_urls']

## Step: `read_raw`
**Op:** `read_raw_files`

Open raw echosounder files with echopype and write each one to an intermediate store. Returns the store paths, or in-memory EchoData objects when intermediate_format is "none".

**Parameters:**
- `sonar_model` str - Sonar model string recognized by echopype (e.g. 'EK60', 'EK80')
- `include_bot` bool - Include bottom-detection (.bot) data if available
- `intermediate_format` str - Store format written between reading and combining. "netcdf" (default, recommended) writes temporary .nc files. "none" keeps everything in memory. "zarr" writes zarr stores, which can live on a gs:// temp dir.

- `use_swap` str - Whether echopype swaps data to disk while reading. "auto" (default) swaps a file whose expanded data would exceed about 40% of total RAM.


In [ ]:
# --- Parameters ---
_read_raw__sonar_model = 'EK60'
_read_raw__include_bot = False

raw_stores = read_raw_files_to_stores(
    raw_file_paths=raw_urls,
    sonar_model=_read_raw__sonar_model,
    include_bot=_read_raw__include_bot,
)

## Step: `combine_raw`
**Op:** `combine_raw_files`

Combine the per-file stores or EchoData objects from read_raw_files into one EchoData object. Mark the step checkpoint: always so the combined output is written once and reused.

**Parameters:**
- `ping_time_chunk` int - Chunk length along ping_time for the combined EchoData and its checkpoint, in pings. Larger values write faster to gs:// but use more memory; try 2000 to 4000 for bucket caches.


In [ ]:
echodata = combine_raw_stores(raw_stores=raw_stores)

## Step: `compute_sv`
**Op:** `compute_sv`

Compute volume backscattering strength (Sv) from raw EchoData, applying cal_params and env_params when given and the raw file's own values otherwise.

In [ ]:
ds_Sv = compute_Sv(echodata=echodata)

## Step: `plot_sv`
**Op:** `plot_sv_echogram`

Render an Sv echogram of ds_Sv, optionally on the coordinate axes of ds_Sv_source (for example MVBS on the original Sv grid). A sink step: it makes a figure and has no data output.

**Parameters:**
- `min_depth` float - Minimum depth (y-axis) in meters
- `max_depth` float - Maximum depth (y-axis) in meters. Omit to use the deepest depth present in the data.

- `ping_min` int - First ping index to display
- `ping_max` int - Last ping index to display. Omit to display through the last ping.
- `time_min` str - Start of the displayed window as a UTC timestamp, for example "2024-10-15T12:30". Resolved to the closest ping and used instead of ping_min. Omit to window by ping index.

- `time_max` str - End of the displayed window as a UTC timestamp. Resolved to the closest ping and used instead of ping_max. Omit to window by ping index.

- `x_axis_units` str - Units for the x-axis: 'datetime' (default, absolute UTC clock times), 'seconds' from the first ping shown, 'pings', 'meters' (distance travelled, derived from the GPS track), or 'bins' (MVBS input only).

- `y_axis_units` str - Units for the y-axis: 'meters' (default), 'range_sample', or 'bins' (MVBS input only).

- `panel_width_to_height` float - Force each frequency panel to be this many times wider than tall (for example 8 for a long strip). Omit for automatic sizing.

- `overlay_lines` list - Optional list of line overlays. Each entry is a dict with 'var' (variable name in ds_Sv) and 'style' (dict of matplotlib kwargs).

- `save_image` bool - Save the rendered figure to an image file; null uses runtime defaults
- `save_formats` list - Image formats to save, chosen from png, svg, jpeg, jpg, and pdf
- `save_dir` str - Optional directory override for saved image artifacts
- `show` bool - Whether to display the figure interactively or inline; null uses runtime defaults
- `dpi` int - Optional DPI override for raster image outputs

In [ ]:
# --- Parameters ---
_plot_sv__max_depth = 1000.0

plot_sv_echogram(ds_Sv=ds_Sv, max_depth=_plot_sv__max_depth)